In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset as TorchDataset, DataLoader
from collections import defaultdict
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler
from typing import List, Dict, Optional, Tuple
import wandb
import os
import copy
import tqdm as tqdm
import math

# -------------------------------------------------------------------
# Part 1: IMPORTS FROM CYBENCH LIBRARY
# -------------------------------------------------------------------
from cybench.config import (
    KEY_LOC,
    KEY_YEAR,
    STATIC_PREDICTORS,
    TIME_SERIES_PREDICTORS,
)
from cybench.datasets.configured import load_dfs_crop

# -------------------------------------------------------------------
# Part 2: EXPERIMENT CONFIGURATION
# -------------------------------------------------------------------
CROPS_TO_RUN = ['maize', 'wheat']
CROP_TO_ID = {"maize": 0, "wheat": 1}

# Restricted to USA only
TRAIN_COUNTRIES = ['US']
TEST_COUNTRIES = ['US']
ALL_COUNTRIES = sorted(list(set(TRAIN_COUNTRIES + TEST_COUNTRIES)))

MODELS_TO_RUN = ['LSTM', 'LSTMRes', 'Transformer', 'TransformerRes']

NN_MODELS_EPOCHS = 10
EARLY_STOPPING_PATIENCE = 5 
SAVE_TOP_K_CHECKPOINTS = 3 
D_MODEL_EMB = 32 
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")
print(f"Running for {NN_MODELS_EPOCHS} epochs with patience {EARLY_STOPPING_PATIENCE}.")
print(f"Models: {MODELS_TO_RUN}")
print(f"Crops: {CROPS_TO_RUN}")

# Globals
N_TIME_SERIES_INPUTS = len(TIME_SERIES_PREDICTORS)
N_STATIC_INPUTS = len(STATIC_PREDICTORS)
MAX_SEQ_LEN = 36 
ADM_TO_ID_MAPPING = {}
NUM_REGIONS = 0

# -------------------------------------------------------------------
# Part 3: HELPER FUNCTIONS (Data Handling)
# -------------------------------------------------------------------

class SeqYieldDataset(TorchDataset):
    """Pytorch dataset holding time-series, targets, crop IDs, region IDs."""
    def __init__(self, X_ts: np.ndarray, X_static: np.ndarray, y: np.ndarray, crops: List[str], region_ids: List[int], years: Optional[List[int]] = None):
        self.X_ts = torch.tensor(X_ts, dtype=torch.float32)
        self.X_static = torch.tensor(X_static, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.crop_ids = torch.tensor([CROP_TO_ID.get(c, 0) for c in crops], dtype=torch.long)
        self.region_ids = torch.tensor(region_ids, dtype=torch.long)
        self.years = torch.tensor(years, dtype=torch.long) if years is not None else None
        
        print(f"[Dataset] Created: X_ts={self.X_ts.shape}, X_static={self.X_static.shape}, y={self.y.shape}, regions={self.region_ids.shape}")

    def __len__(self): return len(self.y)
    
    def __getitem__(self, idx):
        item = (self.X_ts[idx], self.X_static[idx], self.y[idx], self.crop_ids[idx], self.region_ids[idx])
        if self.years is not None:
            return item + (self.years[idx],)
        return item

def prepare_data(
    df_y_subset: pd.DataFrame, 
    dfs_x: Dict[str, pd.DataFrame], 
    adm_to_id_map: Dict[str, int],
    ts_scaler: StandardScaler,
    static_scaler: StandardScaler,
    max_seq_len: int,
    fit_scalers: bool = False
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str], List[int], List[int]]:
    
    X_ts_list, X_static_list, y_list = [], [], []
    crops_list, region_ids_list, years_list = [], [], []
    
    df_static = dfs_x['static'].reindex(columns=STATIC_PREDICTORS)
    if fit_scalers:
        static_scaler.fit(df_static)
    X_static_scaled = static_scaler.transform(df_static)
    df_static_scaled = pd.DataFrame(X_static_scaled, index=df_static.index, columns=df_static.columns)

    df_ts = dfs_x['time_series'].reindex(columns=TIME_SERIES_PREDICTORS)
    all_dekads = list(range(1, 37))
    full_idx = pd.MultiIndex.from_product(
        [df_ts.index.get_level_values(KEY_LOC).unique(), 
         df_ts.index.get_level_values(KEY_YEAR).unique(), 
         all_dekads],
        names=[KEY_LOC, KEY_YEAR, 'dekad']
    )
    df_ts = df_ts.reindex(full_idx)
    df_ts = df_ts.groupby(level=[KEY_LOC, KEY_YEAR]).apply(lambda g: g.interpolate(method='linear', limit_direction='both', axis=0))
    df_ts = df_ts.fillna(0) 
    
    if fit_scalers:
        sample_size = min(500000, len(df_ts))
        ts_scaler.fit(df_ts.sample(sample_size, random_state=42))
        
    X_ts_scaled = ts_scaler.transform(df_ts)
    df_ts_scaled = pd.DataFrame(X_ts_scaled, index=df_ts.index, columns=df_ts.columns)
    
    for (adm_id, year), row in df_y_subset.iterrows():
        if adm_id not in df_static_scaled.index:
            continue
        if (adm_id, year) not in df_ts_scaled.index:
            continue
            
        y_val = row['yield']
        crop_str = row['crop']
        region_id = adm_to_id_map[adm_id]
        
        static_features = df_static_scaled.loc[adm_id].values
        
        try:
            ts_features_df = df_ts_scaled.loc[(adm_id, year)]
        except KeyError:
            continue
            
        ts_features = ts_features_df.values
        if len(ts_features) > max_seq_len:
            ts_features = ts_features[:max_seq_len]
        elif len(ts_features) < max_seq_len:
            pad_width = ((0, max_seq_len - len(ts_features)), (0, 0))
            ts_features = np.pad(ts_features, pad_width, 'constant', constant_values=0)
            
        X_ts_list.append(ts_features)
        X_static_list.append(static_features)
        y_list.append(y_val)
        crops_list.append(crop_str)
        region_ids_list.append(region_id)
        years_list.append(year)

    return (
        np.array(X_ts_list),
        np.array(X_static_list),
        np.array(y_list),
        crops_list,
        region_ids_list,
        years_list
    )

# -------------------------------------------------------------------
# Part 4: CUSTOM BASE MODEL
# -------------------------------------------------------------------

class CustomBaseModel(nn.Module):
    def __init__(self, **kwargs):
        super().__init__()
        self.criterion = nn.MSELoss()
        self.run_name = kwargs.get("run_name", "default_run")
        self.best_checkpoints = []
        self.save_top_k = kwargs.get("save_top_k", 3)
    
    def fit(self, train_loader, val_loader, epochs, device, use_wandb=True, early_stopping_patience=5):
        self.to(device)
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
        
        if use_wandb:
            wandb.watch(self, log='all', log_freq=100)

        best_val_loss = float('inf')
        patience_counter = 0
        self.best_checkpoints = []

        print(f"\n[Fit] Starting training for {epochs} epochs...")
        for epoch in range(1, epochs + 1):
            # --- Training ---
            self.train()
            train_loss = 0.0
            for batch in train_loader:
                batch_on_device = [b.to(device) for b in batch]
                x_ts, x_static, y, crop_id, region_id = batch_on_device[:5]
                
                optimizer.zero_grad()
                y_pred = self(x_ts, x_static, crop_id, region_id)
                loss = self.criterion(y_pred.squeeze(), y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * x_ts.size(0)
            
            train_loss /= len(train_loader.dataset)

            # --- Validation ---
            self.eval()
            val_loss = 0.0
            with torch.no_grad():
                for batch in val_loader:
                    batch_on_device = [b.to(device) for b in batch]
                    x_ts, x_static, y, crop_id, region_id = batch_on_device[:5]
                    
                    y_pred = self(x_ts, x_static, crop_id, region_id)
                    loss = self.criterion(y_pred.squeeze(), y)
                    val_loss += loss.item() * x_ts.size(0)
            
            val_loss /= len(val_loader.dataset)
            val_rmse = np.sqrt(val_loss)

            print(f"Epoch {epoch:02d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val RMSE: {val_rmse:.4f}")

            if use_wandb:
                wandb.log({
                    "epoch": epoch,
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                    "val_rmse": val_rmse
                })

            # --- Early Stopping & Checkpoint Saving ---
            self.best_checkpoints.append((val_loss, copy.deepcopy(self.state_dict())))
            self.best_checkpoints = sorted(self.best_checkpoints, key=lambda x: x[0])[:self.save_top_k]
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
            else:
                patience_counter += 1
            
            if patience_counter >= early_stopping_patience:
                print(f"Stopping early after {patience_counter} epochs with no improvement.")
                break
        
        print(f"[Fit] Training complete. Best Val Loss: {best_val_loss:.4f}")
        print(f"Saved {len(self.best_checkpoints)} checkpoints. Loading best model...")
        self.load_state_dict(self.best_checkpoints[0][1])
        for i, (loss, state_dict) in enumerate(self.best_checkpoints):
            ckpt_path = f"{self.run_name}_ckpt_rank_{i+1}_loss_{loss:.4f}.pth"
            torch.save(state_dict, ckpt_path)
            if use_wandb:
                wandb.save(ckpt_path)

    def predict(self, test_loader, device) -> Tuple[np.ndarray, np.ndarray]:
        self.to(device)
        self.eval()
        all_preds = []
        all_true = []
        with torch.no_grad():
            for batch in test_loader:
                batch_on_device = [b.to(device) for b in batch]
                x_ts, x_static, y, crop_id, region_id = batch_on_device[:5]
                
                y_pred = self(x_ts, x_static, crop_id, region_id)
                
                all_preds.append(y_pred.squeeze().cpu().numpy())
                all_true.append(y.cpu().numpy())
        
        return np.concatenate(all_preds), np.concatenate(all_true)


# -------------------------------------------------------------------
# Part 5: NEURAL NETWORK MODEL IMPLEMENTATIONS
# -------------------------------------------------------------------

class LSTMModel(CustomBaseModel):
    def __init__(self, n_ts_inputs, n_static_inputs, num_regions, d_model, max_seq_len, hidden_size=64, num_layers=1, output_size=1, **kwargs):
        super().__init__(**kwargs)
        self.crop_emb = nn.Embedding(2, d_model)
        self.region_emb = nn.Embedding(num_regions, d_model)
        self.lstm = nn.LSTM(n_ts_inputs, hidden_size, num_layers, batch_first=True, bidirectional=False)
        fc_inputs = hidden_size + n_static_inputs + d_model + d_model
        self.fc = nn.Linear(fc_inputs, output_size)
        self.relu = nn.ReLU()

    def forward(self, x_ts, x_static, crop_ids, region_ids):
        lstm_out, _ = self.lstm(x_ts)
        lstm_feat = lstm_out[:, -1, :] 
        crop_vec = self.crop_emb(crop_ids)
        region_vec = self.region_emb(region_ids)
        combined = torch.cat([lstm_feat, x_static, crop_vec, region_vec], dim=1)
        return self.fc(combined)

class LSTMResModel(CustomBaseModel):
    def __init__(self, n_ts_inputs, n_static_inputs, num_regions, d_model, max_seq_len, hidden_size=64, num_layers=1, output_size=1, **kwargs):
        super().__init__(**kwargs)
        self.crop_emb = nn.Embedding(2, d_model)
        self.region_emb = nn.Embedding(num_regions, d_model)
        self.lstm = nn.LSTM(n_ts_inputs, hidden_size, num_layers, batch_first=True, bidirectional=False)
        fc_inputs = hidden_size + n_static_inputs + d_model + d_model
        self.fc_block1 = nn.Linear(fc_inputs, fc_inputs) 
        self.relu = nn.ReLU()
        self.fc_block2 = nn.Linear(fc_inputs, 50)
        self.fc_final = nn.Linear(50, output_size)

    def forward(self, x_ts, x_static, crop_ids, region_ids):
        lstm_out, _ = self.lstm(x_ts)
        lstm_feat = lstm_out[:, -1, :] 
        crop_vec = self.crop_emb(crop_ids)
        region_vec = self.region_emb(region_ids)
        combined = torch.cat([lstm_feat, x_static, crop_vec, region_vec], dim=1)
        identity = combined
        out = self.relu(self.fc_block1(combined))
        out = out + identity 
        out = self.relu(out)
        out = self.relu(self.fc_block2(out))
        return self.fc_final(out)

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 500, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(1, max_len, d_model)
        pe[0, :, 0::2] = torch.sin(position * div_term)
        pe[0, :, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe) 

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class TransformerModel(CustomBaseModel):
    def __init__(self, n_ts_inputs, n_static_inputs, num_regions, d_model, max_seq_len, nhead=4, num_layers=2, output_size=1, **kwargs):
        super().__init__(**kwargs)
        self.ts_input_proj = nn.Linear(n_ts_inputs, d_model)
        self.crop_emb = nn.Embedding(2, d_model)
        self.region_emb = nn.Embedding(num_regions, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_seq_len)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True, dim_feedforward=d_model*2)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        fc_inputs = d_model + n_static_inputs
        self.fc1 = nn.Linear(fc_inputs, 50)
        self.fc2 = nn.Linear(50, output_size)
        self.relu = nn.ReLU()

    def forward(self, x_ts, x_static, crop_ids, region_ids):
        x_ts_proj = self.ts_input_proj(x_ts) 
        crop_vec = self.crop_emb(crop_ids).unsqueeze(1)
        region_vec = self.region_emb(region_ids).unsqueeze(1) 
        x = x_ts_proj + crop_vec + region_vec 
        x = self.pos_encoder(x) 
        tf_out = self.transformer_encoder(x) 
        tf_feat = tf_out.mean(dim=1) 
        combined = torch.cat([tf_feat, x_static], dim=1)
        out = self.relu(self.fc1(combined))
        return self.fc2(out)

class TransformerResModel(CustomBaseModel):
    def __init__(self, n_ts_inputs, n_static_inputs, num_regions, d_model, max_seq_len, nhead=4, num_layers=2, output_size=1, **kwargs):
        super().__init__(**kwargs)
        self.ts_input_proj = nn.Linear(n_ts_inputs, d_model)
        self.crop_emb = nn.Embedding(2, d_model)
        self.region_emb = nn.Embedding(num_regions, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_seq_len)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True, dim_feedforward=d_model*2)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        fc_inputs = d_model + n_static_inputs
        self.fc_block1 = nn.Linear(fc_inputs, fc_inputs)
        self.relu = nn.ReLU()
        self.fc_block2 = nn.Linear(fc_inputs, 50)
        self.fc_final = nn.Linear(50, output_size)

    def forward(self, x_ts, x_static, crop_ids, region_ids):
        x_ts_proj = self.ts_input_proj(x_ts) 
        crop_vec = self.crop_emb(crop_ids).unsqueeze(1)
        region_vec = self.region_emb(region_ids).unsqueeze(1) 
        x = x_ts_proj + crop_vec + region_vec 
        x = self.pos_encoder(x) 
        tf_out = self.transformer_encoder(x) 
        tf_feat = tf_out.mean(dim=1) 
        combined = torch.cat([tf_feat, x_static], dim=1)
        identity = combined
        out = self.relu(self.fc_block1(combined))
        out = out + identity 
        out = self.relu(out)
        out = self.relu(self.fc_block2(out))
        return self.fc_final(out)

# -------------------------------------------------------------------
# Part 6: METRICS AND DATA LOADING
# -------------------------------------------------------------------

def calculate_metrics(y_true, y_pred, prefix=""):
    if isinstance(y_true, pd.Series): y_true = y_true.values
    if isinstance(y_pred, pd.Series): y_pred = y_pred.values

    mask = y_true != 0
    y_true_safe = y_true[mask]
    y_pred_safe = y_pred[mask]
    
    if len(y_true_safe) == 0:
        return {
            f"{prefix}rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
            f"{prefix}nrmse": np.inf,
            f"{prefix}r2": r2_score(y_true, y_pred),
            f"{prefix}mape": np.inf
        }

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true_safe, y_pred_safe)
    mean_y_true = np.mean(y_true)
    nrmse = np.inf if mean_y_true == 0 else (rmse / mean_y_true)
        
    return {
        f"{prefix}rmse": rmse,
        f"{prefix}nrmse": nrmse,
        f"{prefix}r2": r2,
        f"{prefix}mape": mape
    }

def load_and_combine_data(crops: List[str], countries: List[str]) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
    print(f"\n[Load] Loading data for {crops} in {len(countries)} countries...")
    df_y_list = []
    raw_dfs_from_all_crops = []
    
    for crop in crops:
        print(f"  Loading {crop}...")
        try:
            df_y_c, dfs_x_c = load_dfs_crop(crop=crop, countries=countries)
            df_y_c['crop'] = crop
            df_y_list.append(df_y_c)
            raw_dfs_from_all_crops.append(dfs_x_c)
            
        except Exception as e:
            print(f"Warning: Could not load data for crop {crop}. Error: {e}")
            
    if not df_y_list:
        raise RuntimeError("No data loaded. Exiting.")

    df_y_all = pd.concat(df_y_list).sort_index()
    if not df_y_all.index.is_unique:
        df_y_all = df_y_all.loc[~df_y_all.index.duplicated(keep='first')]
    
    combined_raw_dfs = defaultdict(list)
    for dfs_x_c in raw_dfs_from_all_crops:
        for key, df in dfs_x_c.items():
            if df is not None and not df.empty:
                combined_raw_dfs[key].append(df)
    
    final_raw_dfs = {}
    for key, df_list in combined_raw_dfs.items():
        try:
            concatenated_df = pd.concat(df_list)
            if not concatenated_df.index.is_unique:
                concatenated_df = concatenated_df.loc[~concatenated_df.index.duplicated(keep='first')]
            final_raw_dfs[key] = concatenated_df
        except Exception as e:
            print(f"  Warning: Could not concat raw_df '{key}'. Error: {e}")

    static_dfs_to_join = []
    ts_dfs_to_join = []
    
    for key, df in final_raw_dfs.items():
        if KEY_YEAR in df.index.names:
            ts_dfs_to_join.append(df)
        elif KEY_LOC in df.index.names:
            static_dfs_to_join.append(df)

    all_locs_index = df_y_all.index.get_level_values(KEY_LOC).unique()
    if not all_locs_index.any():
         all_locs_index = pd.Index([], name=KEY_LOC)

    if static_dfs_to_join:
        df_static_all = static_dfs_to_join[0].join(static_dfs_to_join[1:], how='outer')
        df_static_all = df_static_all.loc[:, ~df_static_all.columns.duplicated(keep='first')]
        cols_to_keep = [col for col in STATIC_PREDICTORS if col in df_static_all.columns]
        dfs_x_all_static = df_static_all[cols_to_keep]
        dfs_x_all_static = dfs_x_all_static.reindex(all_locs_index)
    else:
        dfs_x_all_static = pd.DataFrame(index=all_locs_index, columns=STATIC_PREDICTORS)

    dfs_x_all_ts = None
    if ts_dfs_to_join:
        dekad_level_dfs = [df for df in ts_dfs_to_join if df.index.nlevels > 2]
        year_level_dfs = [df for df in ts_dfs_to_join if df.index.nlevels == 2]

        df_ts_dekad = None
        df_ts_year = None
        
        if dekad_level_dfs:
            df_ts_dekad = dekad_level_dfs[0].join(dekad_level_dfs[1:], how='outer')
        if year_level_dfs:
            df_ts_year = year_level_dfs[0].join(year_level_dfs[1:], how='outer')

        if df_ts_dekad is not None and df_ts_year is not None:
            dfs_x_all_ts = df_ts_dekad.join(df_ts_year, how='outer')
        elif df_ts_dekad is not None:
            dfs_x_all_ts = df_ts_dekad
        elif df_ts_year is not None:
            dfs_x_all_ts = df_ts_year
        
        if dfs_x_all_ts is not None:
            dfs_x_all_ts = dfs_x_all_ts.loc[:, ~dfs_x_all_ts.columns.duplicated(keep='first')]
            cols_to_keep = [col for col in TIME_SERIES_PREDICTORS if col in dfs_x_all_ts.columns]
            dfs_x_all_ts = dfs_x_all_ts[cols_to_keep]

    if dfs_x_all_ts is None:
        base_idx = df_y_all.index.unique()
        if base_idx.any():
            dfs_x_all_ts = pd.DataFrame(index=base_idx, columns=TIME_SERIES_PREDICTORS)
        else:
            dfs_x_all_ts = pd.DataFrame(columns=TIME_SERIES_PREDICTORS)
            dfs_x_all_ts.index = pd.MultiIndex(levels=[[],[]], codes=[[],[]], names=[KEY_LOC, KEY_YEAR])

    dfs_x_all = {
        'static': dfs_x_all_static,
        'time_series': dfs_x_all_ts
    }
    
    global MAX_SEQ_LEN, ADM_TO_ID_MAPPING, NUM_REGIONS
    
    if not dfs_x_all['time_series'].empty and dfs_x_all['time_series'].index.nlevels > 2:
        group_levels = dfs_x_all['time_series'].index.names[:-1]
        dekads_per_sample = dfs_x_all['time_series'].groupby(level=group_levels).size()
        
        if not dekads_per_sample.empty:
            MAX_SEQ_LEN = dekads_per_sample.max()
            print(f"[Load] Max sequence length (dekads) found: {MAX_SEQ_LEN}")
    
    unique_adms = df_y_all.index.get_level_values(KEY_LOC).unique()
    ADM_TO_ID_MAPPING = {adm: i for i, adm in enumerate(unique_adms)}
    NUM_REGIONS = len(unique_adms)
    print(f"[Load] Found {NUM_REGIONS} unique regions (adm_id).")
    
    return df_y_all, dfs_x_all

# -------------------------------------------------------------------
# Part 7: MAIN EXPERIMENT FUNCTION
# -------------------------------------------------------------------

def run_experiment(model_name: str, crop: str, run_name: str):
    """
    Trains a model for a SINGLE crop using a strict temporal split.
    """
    if model_name not in MODELS_TO_RUN:
        raise ValueError(f"Model '{model_name}' not recognized.")
        
    print("="*60)
    print(f"🚀 STARTING EXPERIMENT | Model: {model_name} | Crop: {crop.upper()} | Run: {run_name}")
    print("="*60)

    # --- 1. Load Data for SINGLE Crop ---
    df_y_all, dfs_x_all = load_and_combine_data([crop], ALL_COUNTRIES)
    df_y_all = df_y_all.reset_index() 
    
    # --- 2. Create Temporal Split (Train/Val/Test) ---
    print("\n[Split] Creating strict temporal split...")
    
    years = sorted(df_y_all[KEY_YEAR].unique())
    if len(years) < 6:
        raise ValueError(f"Not enough years ({len(years)}) to create a 3-year test / 3-year val split.")

    # Last 3 years for Test
    test_years = years[-3:]
    # 3 years before that for Validation
    val_years = years[-6:-3]
    # Everything else for Train
    train_years = years[:-6]

    print(f" Train Years: {train_years}")
    print(f" Val Years:   {val_years}")
    print(f" Test Years:  {test_years}")

    df_y_train = df_y_all[df_y_all[KEY_YEAR].isin(train_years)].set_index([KEY_LOC, KEY_YEAR])
    df_y_val = df_y_all[df_y_all[KEY_YEAR].isin(val_years)].set_index([KEY_LOC, KEY_YEAR])
    df_y_test = df_y_all[df_y_all[KEY_YEAR].isin(test_years)].set_index([KEY_LOC, KEY_YEAR])
    
    print(f"[Split] Total samples: {len(df_y_all)}")
    print(f"[Split] Train samples: {len(df_y_train)}")
    print(f"[Split] Val samples:   {len(df_y_val)}")
    print(f"[Split] Test samples:  {len(df_y_test)}")
    
    # --- 3. Prepare Data & Create Datasets ---
    print("\n[Data] Preparing and normalizing data...")
    ts_scaler = StandardScaler()
    static_scaler = StandardScaler()
    
    X_ts_train, X_static_train, y_train, crops_train, regions_train, _ = prepare_data(
        df_y_train, dfs_x_all, ADM_TO_ID_MAPPING, 
        ts_scaler, static_scaler, MAX_SEQ_LEN, fit_scalers=True
    )
    train_dataset = SeqYieldDataset(X_ts_train, X_static_train, y_train, crops_train, regions_train)
    
    X_ts_val, X_static_val, y_val, crops_val, regions_val, _ = prepare_data(
        df_y_val, dfs_x_all, ADM_TO_ID_MAPPING, 
        ts_scaler, static_scaler, MAX_SEQ_LEN, fit_scalers=False
    )
    val_dataset = SeqYieldDataset(X_ts_val, X_static_val, y_val, crops_val, regions_val)
    
    X_ts_test, X_static_test, y_test, crops_test, regions_test, years_test = prepare_data(
        df_y_test, dfs_x_all, ADM_TO_ID_MAPPING, 
        ts_scaler, static_scaler, MAX_SEQ_LEN, fit_scalers=False
    )
    
    test_adm_ids = df_y_test.index.get_level_values(KEY_LOC)
    df_test_eval = pd.DataFrame({
        'adm_id': test_adm_ids,
        'year': years_test,
        'crop': crops_test,
        'region_id': regions_test,
        'y_true': y_test
    })
    df_test_eval['country'] = df_test_eval['adm_id'].str[:2].str.upper()

    test_dataset = SeqYieldDataset(X_ts_test, X_static_test, y_test, crops_test, regions_test, years_test)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # --- 4. Initialize Model ---
    model_init_kwargs = {
        "n_ts_inputs": N_TIME_SERIES_INPUTS,
        "n_static_inputs": N_STATIC_INPUTS,
        "num_regions": NUM_REGIONS,
        "d_model": D_MODEL_EMB,
        "max_seq_len": MAX_SEQ_LEN,
        "run_name": run_name,
        "save_top_k": SAVE_TOP_K_CHECKPOINTS
    }
    
    MODEL_CONSTRUCTORS = {
        "LSTM": LSTMModel,
        "LSTMRes": LSTMResModel,
        "Transformer": TransformerModel,
        "TransformerRes": TransformerResModel,
    }

    print(f"\n[Model] Initializing {model_name}...")
    model = MODEL_CONSTRUCTORS[model_name](**model_init_kwargs)
    
    # --- 5. Train Model ---
    model.fit(
        train_loader, 
        val_loader, 
        epochs=NN_MODELS_EPOCHS, 
        device=DEVICE,
        use_wandb=True,
        early_stopping_patience=EARLY_STOPPING_PATIENCE
    )
    
    # --- 6. Evaluate Model ---
    print("\n[Evaluate] Evaluating model on test set...")
    y_pred, y_true = model.predict(test_loader, DEVICE)
    df_test_eval['y_pred'] = y_pred
    
    # --- 7. Save Detailed Results ---
    print("[Evaluate] Saving detailed evaluation CSVs...")
    
    raw_csv_path = f"{run_name}_predictions_raw.csv"
    df_test_eval.to_csv(raw_csv_path, index=False)
    
    def evaluate_group(group, group_cols):
        metrics = calculate_metrics(group['y_true'], group['y_pred'])
        if isinstance(group_cols, str):
            metrics[group_cols] = group.name
        elif len(group_cols) > 1:
            for i, col in enumerate(group_cols):
                metrics[col] = group.name[i]
        return pd.Series(metrics)

    results_per_year = df_test_eval.groupby('year').apply(
        lambda g: evaluate_group(g, 'year')
    ).reset_index(drop=True)
    results_per_year.to_csv(f"{run_name}_results_per_year.csv", index=False)

    results_overall = pd.Series(calculate_metrics(df_test_eval['y_true'], df_test_eval['y_pred']))
    results_overall['model'] = model_name
    results_overall['crop'] = crop
    results_overall['run_name'] = run_name
    results_overall_df = results_overall.to_frame().T
    results_overall_df.to_csv(f"{run_name}_results_overall.csv", index=False)
    
    print("\n[Evaluate] Overall Test Metrics:")
    print(results_overall_df.to_string(index=False))
    
    if wandb.run:
        wandb.log({"test_metrics_overall": results_overall.to_dict()})
        wandb.log({"test_results_per_year": wandb.Table(dataframe=results_per_year)})

    print("\n" + "="*66)
    print(f"✅ EXPERIMENT COMPLETE | {model_name} on {crop.upper()}")
    print("="*66 + "\n")
    
    return results_overall.to_dict()

# -------------------------------------------------------------------
# Part 8: RUN ALL EXPERIMENTS AND COLLECT RESULTS
# -------------------------------------------------------------------

if __name__ == '__main__':
    os.makedirs("results", exist_ok=True)
    os.environ["WANDB_LOG_SYSTEM"] = "true"
    
    all_results = []

    # Loop over CROPS first, then MODELS
    for crop in CROPS_TO_RUN:
        for model_name in tqdm.tqdm(MODELS_TO_RUN, desc=f"Models for {crop}"):
            
            run_name = f"{model_name}_{crop}_US_temporal_{NN_MODELS_EPOCHS}e"
            
            try:
                wandb.init(
                    project="cybench-LSTM-NEW-3yr", 
                    name=run_name,
                    config={
                        "model": model_name,
                        "crop": crop,
                        "epochs": NN_MODELS_EPOCHS,
                        "batch_size": BATCH_SIZE,
                        "data_split": "temporal_last_3yr_test_3yr_val", 
                        "embedding_dim": D_MODEL_EMB,
                        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
                        "save_top_k": SAVE_TOP_K_CHECKPOINTS
                    },
                    reinit=True
                )
                
                overall_metrics = run_experiment(model_name=model_name, crop=crop, run_name=f"results/{run_name}")
                all_results.append(overall_metrics)
                
            except Exception as e:
                print(f"\n" + "!"*60)
                print(f"🚨 EXPERIMENT FAILED FOR {model_name} on {crop} 🚨")
                print(f"Error: {e}")
                import traceback
                traceback.print_exc()
                print("!"*60 + "\n")
            finally:
                if wandb.run:
                    wandb.finish()

    results_df = pd.DataFrame(all_results)
    
    print("\n" + "#"*25 + " FINAL SUMMARY " + "#"*25)
    print("All experiments completed using separate models per crop and temporal splits (US only).")
    print("#"*65 + "\n")
    
    print(results_df.to_string(index=False))
    
    summary_path = f"results/temporal_summary_results_US_{NN_MODELS_EPOCHS}e.csv"
    results_df.to_csv(summary_path, index=False)
    print(f"\nSummary results saved to {summary_path}")

Using device: cpu
Running for 10 epochs with patience 5.
Models: ['LSTM', 'LSTMRes', 'Transformer', 'TransformerRes']
Crops: ['maize', 'wheat']


Models for maize:   0%|          | 0/4 [00:00<?, ?it/s]wandb: Currently logged in as: swarang (swarang-maastricht-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


🚀 STARTING EXPERIMENT | Model: LSTM | Crop: MAIZE | Run: results/LSTM_maize_US_temporal_10e

[Load] Loading data for ['maize'] in 1 countries...
  Loading maize...
[Load] Max sequence length (dekads) found: 199
[Load] Found 2221 unique regions (adm_id).

[Split] Creating strict temporal split...
 Train Years: [np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]
 Val Years:   [np.int64(2018), np.int64(2019), np.int64(2020)]
 Test Years:  [np.int64(2021), np.int64(2022), np.int64(2023)]
[Split] Total samples: 33193
[Split] Train samples: 24580
[Split] Val samples:   4243
[Split] Test samples:  4370

[Data] Preparing and normalizing data...
[Dataset] Created: X_ts=torch.Size([24580, 199, 9]), X_static=torch.Size([24580, 2]), y=torch.Size([24580]), regions=torch.Size([24580])
[Dataset] Created: X_ts=torch

/var/folders/5v/801lr_c574l81hvxwx6m27pr0000gn/T/ipykernel_62326/2972045242.py:671: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  results_per_year = df_test_eval.groupby('year').apply(
wandb: ERROR The nbformat package was not found. It is required to save notebook history.



✅ EXPERIMENT COMPLETE | LSTM on MAIZE



...

/var/folders/5v/801lr_c574l81hvxwx6m27pr0000gn/T/ipykernel_62326/2972045242.py:671: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  results_per_year = df_test_eval.groupby('year').apply(
wandb: ERROR The nbformat package was not found. It is required to save notebook history.



✅ EXPERIMENT COMPLETE | TransformerRes on WHEAT



epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▅▃▂▁▁▁▁▁▁
val_loss,█▆▂▁▁▁▁▁▁▁
val_rmse,█▆▃▂▁▂▂▁▂▁
epoch,10
train_loss,0.44238
val_loss,0.46095
val_rmse,0.67893


Models for wheat: 100%|██████████| 4/4 [3:04:23<00:00, 2765.77s/it]


######################### FINAL SUMMARY #########################
All experiments completed using separate models per crop and temporal splits (US only).
#################################################################

    rmse    nrmse       r2     mape          model  crop                                     run_name
2.142902 0.212286 0.251772 0.196306           LSTM maize           results/LSTM_maize_US_temporal_10e
2.063220 0.204392 0.306382 0.192019        LSTMRes maize        results/LSTMRes_maize_US_temporal_10e
2.013705 0.199487 0.339275 0.186767    Transformer maize    results/Transformer_maize_US_temporal_10e
1.983191 0.196464 0.359147 0.183080 TransformerRes maize results/TransformerRes_maize_US_temporal_10e
1.038735 0.245318 0.534607 0.223460           LSTM wheat           results/LSTM_wheat_US_temporal_10e
0.963169 0.227472 0.599856 0.216324        LSTMRes wheat        results/LSTMRes_wheat_US_temporal_10e
0.935458 0.220927 0.622550 0.211266    Transformer wheat    resu

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset as TorchDataset, DataLoader
from collections import defaultdict
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler
from typing import List, Dict, Optional, Tuple
import wandb
import os
import copy
import tqdm as tqdm

# -------------------------------------------------------------------
# Part 1: IMPORTS FROM CYBENCH LIBRARY
# -------------------------------------------------------------------
from cybench.config import (
    KEY_LOC,
    KEY_YEAR,
    STATIC_PREDICTORS,
    TIME_SERIES_PREDICTORS,
)
from cybench.datasets.configured import load_dfs_crop

# -------------------------------------------------------------------
# Part 2: NEW & UPDATED EXPERIMENT CONFIGURATION
# -------------------------------------------------------------------
CROPS_TO_RUN = ['maize', 'wheat']
CROP_TO_ID = {"maize": 0, "wheat": 1}

# Restricted to USA only
TRAIN_COUNTRIES = ['US']
TEST_COUNTRIES = ['US']
ALL_COUNTRIES = sorted(list(set(TRAIN_COUNTRIES + TEST_COUNTRIES)))

MODELS_TO_RUN = ['CNN', 'BiLSTM', 'SerialCNNBiLSTM', 'ParallelCNNBiLSTM']

NN_MODELS_EPOCHS = 10 
EARLY_STOPPING_PATIENCE = 5 
SAVE_TOP_K_CHECKPOINTS = 3 
D_MODEL_EMB = 32 
BATCH_SIZE = 32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using device: {DEVICE}")
print(f"Running for {NN_MODELS_EPOCHS} epochs with patience {EARLY_STOPPING_PATIENCE}.")
print(f"Models: {MODELS_TO_RUN}")
print(f"Crops: {CROPS_TO_RUN}")

# Globals to be populated after data loading
N_TIME_SERIES_INPUTS = len(TIME_SERIES_PREDICTORS)
N_STATIC_INPUTS = len(STATIC_PREDICTORS)
MAX_SEQ_LEN = 36 
ADM_TO_ID_MAPPING = {}
NUM_REGIONS = 0

# -------------------------------------------------------------------
# Part 3: HELPER FUNCTIONS (Data Handling & Splitting)
# -------------------------------------------------------------------

class SeqYieldDataset(TorchDataset):
    """Pytorch dataset holding time-series, targets, crop IDs, region IDs."""
    def __init__(self, X_ts: np.ndarray, X_static: np.ndarray, y: np.ndarray, crops: List[str], region_ids: List[int], years: Optional[List[int]] = None):
        self.X_ts = torch.tensor(X_ts, dtype=torch.float32)
        self.X_static = torch.tensor(X_static, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.crop_ids = torch.tensor([CROP_TO_ID.get(c, 0) for c in crops], dtype=torch.long)
        self.region_ids = torch.tensor(region_ids, dtype=torch.long)
        self.years = torch.tensor(years, dtype=torch.long) if years is not None else None
        
        print(f"[Dataset] Created: X_ts={self.X_ts.shape}, X_static={self.X_static.shape}, y={self.y.shape}, regions={self.region_ids.shape}")

    def __len__(self): return len(self.y)
    
    def __getitem__(self, idx):
        item = (self.X_ts[idx], self.X_static[idx], self.y[idx], self.crop_ids[idx], self.region_ids[idx])
        if self.years is not None:
            return item + (self.years[idx],)
        return item

def prepare_data(
    df_y_subset: pd.DataFrame, 
    dfs_x: Dict[str, pd.DataFrame], 
    adm_to_id_map: Dict[str, int],
    ts_scaler: StandardScaler,
    static_scaler: StandardScaler,
    max_seq_len: int,
    fit_scalers: bool = False
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str], List[int], List[int]]:
    
    X_ts_list, X_static_list, y_list = [], [], []
    crops_list, region_ids_list, years_list = [], [], []
    
    # Get normalized static data
    df_static = dfs_x['static'].reindex(columns=STATIC_PREDICTORS)
    if fit_scalers:
        static_scaler.fit(df_static)
    X_static_scaled = static_scaler.transform(df_static)
    df_static_scaled = pd.DataFrame(X_static_scaled, index=df_static.index, columns=df_static.columns)

    # Get normalized time-series data
    df_ts = dfs_x['time_series'].reindex(columns=TIME_SERIES_PREDICTORS)
    all_dekads = list(range(1, 37))
    full_idx = pd.MultiIndex.from_product(
        [df_ts.index.get_level_values(KEY_LOC).unique(), 
         df_ts.index.get_level_values(KEY_YEAR).unique(), 
         all_dekads],
        names=[KEY_LOC, KEY_YEAR, 'dekad']
    )
    df_ts = df_ts.reindex(full_idx)
    df_ts = df_ts.groupby(level=[KEY_LOC, KEY_YEAR]).apply(lambda g: g.interpolate(method='linear', limit_direction='both', axis=0))
    df_ts = df_ts.fillna(0) 
    
    if fit_scalers:
        sample_size = len(df_ts)
        ts_scaler.fit(df_ts.sample(sample_size, random_state=42))
        
    X_ts_scaled = ts_scaler.transform(df_ts)
    df_ts_scaled = pd.DataFrame(X_ts_scaled, index=df_ts.index, columns=df_ts.columns)
    
    for (adm_id, year), row in df_y_subset.iterrows():
        if adm_id not in df_static_scaled.index:
            continue
        if (adm_id, year) not in df_ts_scaled.index:
            continue
            
        y_val = row['yield']
        crop_str = row['crop']
        region_id = adm_to_id_map[adm_id]
        
        static_features = df_static_scaled.loc[adm_id].values
        ts_features_df = df_ts_scaled.loc[(adm_id, year)]
        
        # Pad/truncate TS features
        ts_features = ts_features_df.values
        if len(ts_features) > max_seq_len:
            ts_features = ts_features[:max_seq_len] 
        elif len(ts_features) < max_seq_len:
            pad_width = ((0, max_seq_len - len(ts_features)), (0, 0))
            ts_features = np.pad(ts_features, pad_width, 'constant', constant_values=0)
            
        X_ts_list.append(ts_features)
        X_static_list.append(static_features)
        y_list.append(y_val)
        crops_list.append(crop_str)
        region_ids_list.append(region_id)
        years_list.append(year)

    return (
        np.array(X_ts_list),
        np.array(X_static_list),
        np.array(y_list),
        crops_list,
        region_ids_list,
        years_list
    )

# -------------------------------------------------------------------
# Part 4: NEW CUSTOM BASE MODEL
# -------------------------------------------------------------------

class CustomBaseModel(nn.Module):
    def __init__(self, **kwargs):
        super().__init__()
        self.criterion = nn.MSELoss()
        self.run_name = kwargs.get("run_name", "default_run")
        self.best_checkpoints = [] 
        self.save_top_k = kwargs.get("save_top_k", 3)
    
    def fit(self, train_loader, val_loader, epochs, device, use_wandb=True, early_stopping_patience=5):
        self.to(device)
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
        
        if use_wandb:
            wandb.watch(self, log='all', log_freq=100)

        best_val_loss = float('inf')
        patience_counter = 0
        self.best_checkpoints = []

        print(f"\n[Fit] Starting training for {epochs} epochs...")
        for epoch in range(1, epochs + 1):
            # --- Training ---
            self.train()
            train_loss = 0.0
            for batch in train_loader:
                batch_on_device = [b.to(device) for b in batch]
                x_ts, x_static, y, crop_id, region_id = batch_on_device[:5]
                
                optimizer.zero_grad()
                y_pred = self(x_ts, x_static, crop_id, region_id)
                loss = self.criterion(y_pred.squeeze(), y)
                loss.backward()
                optimizer.step()
                train_loss += loss.item() * x_ts.size(0)
            
            train_loss /= len(train_loader.dataset)

            # --- Validation ---
            self.eval()
            val_loss = 0.0
            with torch.no_grad():
                for batch in val_loader:
                    batch_on_device = [b.to(device) for b in batch]
                    x_ts, x_static, y, crop_id, region_id = batch_on_device[:5]
                    
                    y_pred = self(x_ts, x_static, crop_id, region_id)
                    loss = self.criterion(y_pred.squeeze(), y)
                    val_loss += loss.item() * x_ts.size(0)
            
            val_loss /= len(val_loader.dataset)
            val_rmse = np.sqrt(val_loss)

            print(f"Epoch {epoch:02d}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val RMSE: {val_rmse:.4f}")

            if use_wandb:
                wandb.log({
                    "epoch": epoch,
                    "train_loss": train_loss,
                    "val_loss": val_loss,
                    "val_rmse": val_rmse
                })

            # --- Early Stopping & Checkpoint Saving ---
            self.best_checkpoints.append((val_loss, copy.deepcopy(self.state_dict())))
            self.best_checkpoints = sorted(self.best_checkpoints, key=lambda x: x[0])[:self.save_top_k]
            
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
            else:
                patience_counter += 1
            
            if patience_counter >= early_stopping_patience:
                print(f"Stopping early after {patience_counter} epochs with no improvement.")
                break
        
        print(f"[Fit] Training complete. Best Val Loss: {best_val_loss:.4f}")
        print(f"Saved {len(self.best_checkpoints)} checkpoints. Loading best model...")
        self.load_state_dict(self.best_checkpoints[0][1])
        for i, (loss, state_dict) in enumerate(self.best_checkpoints):
            ckpt_path = f"{self.run_name}_ckpt_rank_{i+1}_loss_{loss:.4f}.pth"
            torch.save(state_dict, ckpt_path)
            if use_wandb:
                wandb.save(ckpt_path)

    def predict(self, test_loader, device) -> Tuple[np.ndarray, np.ndarray]:
        self.to(device)
        self.eval()
        all_preds = []
        all_true = []
        with torch.no_grad():
            for batch in test_loader:
                batch_on_device = [b.to(device) for b in batch]
                x_ts, x_static, y, crop_id, region_id = batch_on_device[:5]
                
                y_pred = self(x_ts, x_static, crop_id, region_id)
                
                all_preds.append(y_pred.squeeze().cpu().numpy())
                all_true.append(y.cpu().numpy())
        
        return np.concatenate(all_preds), np.concatenate(all_true)


# -------------------------------------------------------------------
# Part 5: NEURAL NETWORK MODEL IMPLEMENTATIONS
# -------------------------------------------------------------------

class CNNModel(CustomBaseModel):
    """A 1D CNN model, modified for crop/region embeddings."""
    def __init__(self, n_ts_inputs, n_static_inputs, num_regions, d_model, max_seq_len, output_size=1, cnn_filters=64, **kwargs):
        super().__init__(**kwargs)
        self.crop_emb = nn.Embedding(2, d_model)
        self.region_emb = nn.Embedding(num_regions, d_model)
        
        self.conv1 = nn.Conv1d(in_channels=n_ts_inputs, out_channels=cnn_filters, kernel_size=3, padding=1)
        self.adaptive_pool = nn.AdaptiveMaxPool1d(output_size=5)
        self.relu = nn.ReLU()

        fc1_inputs = (cnn_filters * 5) + n_static_inputs + d_model + d_model
        
        self.fc1 = nn.Linear(fc1_inputs, 50)
        self.fc2 = nn.Linear(50, output_size)

    def forward(self, x_ts, x_static, crop_ids, region_ids):
        x_ts = x_ts.permute(0, 2, 1) 
        x_ts = self.relu(self.conv1(x_ts))
        x_ts = self.adaptive_pool(x_ts)
        x_ts = torch.flatten(x_ts, 1) 
        
        crop_vec = self.crop_emb(crop_ids)    
        region_vec = self.region_emb(region_ids) 
        
        combined = torch.cat([x_ts, x_static, crop_vec, region_vec], dim=1)
        out = self.relu(self.fc1(combined))
        return self.fc2(out)

class BiLSTMModel(CustomBaseModel):
    """A Bidirectional LSTM model, modified for crop/region embeddings."""
    def __init__(self, n_ts_inputs, n_static_inputs, num_regions, d_model, max_seq_len, hidden_size=64, num_layers=1, output_size=1, **kwargs):
        super().__init__(**kwargs)
        self.crop_emb = nn.Embedding(2, d_model)
        self.region_emb = nn.Embedding(num_regions, d_model)
        
        self.lstm = nn.LSTM(n_ts_inputs, hidden_size, num_layers, batch_first=True, bidirectional=True)
        fc_inputs = (hidden_size * 2) + n_static_inputs + d_model + d_model
        self.fc = nn.Linear(fc_inputs, output_size)

    def forward(self, x_ts, x_static, crop_ids, region_ids):
        lstm_out, _ = self.lstm(x_ts)
        lstm_feat = lstm_out[:, -1, :] 
        
        crop_vec = self.crop_emb(crop_ids)    
        region_vec = self.region_emb(region_ids) 
        
        combined = torch.cat([lstm_feat, x_static, crop_vec, region_vec], dim=1)
        return self.fc(combined)

class SerialCNNBiLSTM(CustomBaseModel):
    """A serial hybrid, modified for crop/region embeddings."""
    def __init__(self, n_ts_inputs, n_static_inputs, num_regions, d_model, max_seq_len, cnn_filters=32, lstm_hidden=64, output_size=1, **kwargs):
        super().__init__(**kwargs)
        self.crop_emb = nn.Embedding(2, d_model)
        self.region_emb = nn.Embedding(num_regions, d_model)
        
        self.conv1 = nn.Conv1d(in_channels=n_ts_inputs, out_channels=cnn_filters, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.lstm = nn.LSTM(cnn_filters, lstm_hidden, num_layers=1, batch_first=True, bidirectional=True)
        
        fc_inputs = (lstm_hidden * 2) + n_static_inputs + d_model + d_model
        self.fc = nn.Linear(fc_inputs, output_size)

    def forward(self, x_ts, x_static, crop_ids, region_ids):
        x_ts = x_ts.permute(0, 2, 1) 
        cnn_out = self.relu(self.conv1(x_ts))
        cnn_out = cnn_out.permute(0, 2, 1) 
        
        lstm_out, _ = self.lstm(cnn_out)
        lstm_feat = lstm_out[:, -1, :] 
        
        crop_vec = self.crop_emb(crop_ids)    
        region_vec = self.region_emb(region_ids) 
        
        combined = torch.cat([lstm_feat, x_static, crop_vec, region_vec], dim=1)
        return self.fc(combined)

class ParallelCNNBiLSTM(CustomBaseModel):
    """A parallel hybrid, modified for crop/region embeddings."""
    def __init__(self, n_ts_inputs, n_static_inputs, num_regions, d_model, max_seq_len, cnn_filters=32, lstm_hidden=32, output_size=1, **kwargs):
        super().__init__(**kwargs)
        self.crop_emb = nn.Embedding(2, d_model)
        self.region_emb = nn.Embedding(num_regions, d_model)
        
        self.conv1 = nn.Conv1d(in_channels=n_ts_inputs, out_channels=cnn_filters, kernel_size=3, padding=1)
        self.adaptive_pool = nn.AdaptiveMaxPool1d(output_size=5)
        
        self.lstm = nn.LSTM(n_ts_inputs, lstm_hidden, num_layers=1, batch_first=True, bidirectional=True)
        
        fc_inputs = (cnn_filters * 5) + (lstm_hidden * 2) + n_static_inputs + d_model + d_model
        self.fc = nn.Linear(fc_inputs, output_size)
        self.relu = nn.ReLU()

    def forward(self, x_ts, x_static, crop_ids, region_ids):
        x_ts_cnn = x_ts.permute(0, 2, 1)
        cnn_out = self.relu(self.conv1(x_ts_cnn))
        cnn_out = self.adaptive_pool(cnn_out)
        cnn_out = torch.flatten(cnn_out, 1) 
        
        lstm_out, _ = self.lstm(x_ts)
        lstm_feat = lstm_out[:, -1, :] 
        
        crop_vec = self.crop_emb(crop_ids)    
        region_vec = self.region_emb(region_ids) 
        
        combined = torch.cat([cnn_out, lstm_feat, x_static, crop_vec, region_vec], dim=1)
        return self.fc(combined)

# -------------------------------------------------------------------
# Part 6: METRICS AND DATA LOADING
# -------------------------------------------------------------------

def calculate_metrics(y_true, y_pred, prefix=""):
    """Calculates RMSE, nRMSE, R², MAPE."""
    if isinstance(y_true, pd.Series): y_true = y_true.values
    if isinstance(y_pred, pd.Series): y_pred = y_pred.values

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    mean_y_true = np.mean(y_true)
    if mean_y_true == 0:
        nrmse = np.inf
    else:
        nrmse = (rmse / mean_y_true)
        
    return {
        f"{prefix}rmse": rmse,
        f"{prefix}nrmse": nrmse,
        f"{prefix}r2": r2,
        f"{prefix}mape": mape
    }

def load_and_combine_data(crops: List[str], countries: List[str]) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
    print(f"\n[Load] Loading data for {crops} in {len(countries)} countries...")
    df_y_list = []
    raw_dfs_from_all_crops = []
    
    for crop in crops:
        print(f"  Loading {crop}...")
        try:
            df_y_c, dfs_x_c = load_dfs_crop(crop=crop, countries=countries)
            df_y_c['crop'] = crop 
            df_y_list.append(df_y_c)
            raw_dfs_from_all_crops.append(dfs_x_c)
            
        except Exception as e:
            print(f"Warning: Could not load data for crop {crop}. Error: {e}")
            
    if not df_y_list:
        raise RuntimeError("No data loaded. Exiting.")

    df_y_all = pd.concat(df_y_list).sort_index()
    if not df_y_all.index.is_unique:
        df_y_all = df_y_all.loc[~df_y_all.index.duplicated(keep='first')]
    
    combined_raw_dfs = defaultdict(list)
    for dfs_x_c in raw_dfs_from_all_crops:
        for key, df in dfs_x_c.items():
            if df is not None and not df.empty:
                combined_raw_dfs[key].append(df)
    
    final_raw_dfs = {}
    for key, df_list in combined_raw_dfs.items():
        try:
            concatenated_df = pd.concat(df_list)
            if not concatenated_df.index.is_unique:
                concatenated_df = concatenated_df.loc[~concatenated_df.index.duplicated(keep='first')]
            final_raw_dfs[key] = concatenated_df
        except Exception as e:
            print(f"  Warning: Could not concat raw_df '{key}'. Error: {e}")

    static_dfs_to_join = []
    ts_dfs_to_join = []
    
    for key, df in final_raw_dfs.items():
        if KEY_YEAR in df.index.names:
            ts_dfs_to_join.append(df)
        elif KEY_LOC in df.index.names:
            static_dfs_to_join.append(df)

    all_locs_index = df_y_all.index.get_level_values(KEY_LOC).unique()
    if not all_locs_index.any():
         all_locs_index = pd.Index([], name=KEY_LOC)

    if static_dfs_to_join:
        df_static_all = static_dfs_to_join[0].join(static_dfs_to_join[1:], how='outer')
        df_static_all = df_static_all.loc[:, ~df_static_all.columns.duplicated(keep='first')]
        cols_to_keep = [col for col in STATIC_PREDICTORS if col in df_static_all.columns]
        dfs_x_all_static = df_static_all[cols_to_keep]
        dfs_x_all_static = dfs_x_all_static.reindex(all_locs_index)
    else:
        dfs_x_all_static = pd.DataFrame(index=all_locs_index, columns=STATIC_PREDICTORS)

    dfs_x_all_ts = None
    if ts_dfs_to_join:
        dekad_level_dfs = [df for df in ts_dfs_to_join if df.index.nlevels > 2]
        year_level_dfs = [df for df in ts_dfs_to_join if df.index.nlevels == 2]

        df_ts_dekad = None
        df_ts_year = None
        
        if dekad_level_dfs:
            df_ts_dekad = dekad_level_dfs[0].join(dekad_level_dfs[1:], how='outer')
            
        if year_level_dfs:
            df_ts_year = year_level_dfs[0].join(year_level_dfs[1:], how='outer')

        if df_ts_dekad is not None and df_ts_year is not None:
            dfs_x_all_ts = df_ts_dekad.join(df_ts_year, how='outer')
        elif df_ts_dekad is not None:
            dfs_x_all_ts = df_ts_dekad
        elif df_ts_year is not None:
            dfs_x_all_ts = df_ts_year
        
        if dfs_x_all_ts is not None:
            dfs_x_all_ts = dfs_x_all_ts.loc[:, ~dfs_x_all_ts.columns.duplicated(keep='first')]
            cols_to_keep = [col for col in TIME_SERIES_PREDICTORS if col in dfs_x_all_ts.columns]
            dfs_x_all_ts = dfs_x_all_ts[cols_to_keep]

    if dfs_x_all_ts is None:
        base_idx = df_y_all.index.unique()
        if base_idx.any():
            dfs_x_all_ts = pd.DataFrame(index=base_idx, columns=TIME_SERIES_PREDICTORS)
        else:
            dfs_x_all_ts = pd.DataFrame(columns=TIME_SERIES_PREDICTORS)
            dfs_x_all_ts.index = pd.MultiIndex(levels=[[],[]], codes=[[],[]], names=[KEY_LOC, KEY_YEAR])

    dfs_x_all = {
        'static': dfs_x_all_static,
        'time_series': dfs_x_all_ts
    }
    
    global MAX_SEQ_LEN, ADM_TO_ID_MAPPING, NUM_REGIONS
    
    if not dfs_x_all['time_series'].empty and dfs_x_all['time_series'].index.nlevels > 2:
        group_levels = dfs_x_all['time_series'].index.names[:-1]
        dekads_per_sample = dfs_x_all['time_series'].groupby(level=group_levels).size()
        
        if not dekads_per_sample.empty:
            MAX_SEQ_LEN = dekads_per_sample.max()
            print(f"[Load] Max sequence length (dekads) found: {MAX_SEQ_LEN}")
            
    unique_adms = df_y_all.index.get_level_values(KEY_LOC).unique()
    ADM_TO_ID_MAPPING = {adm: i for i, adm in enumerate(unique_adms)}
    NUM_REGIONS = len(unique_adms)
    print(f"[Load] Found {NUM_REGIONS} unique regions (adm_id).")
    
    return df_y_all, dfs_x_all

# -------------------------------------------------------------------
# Part 7: MAIN EXPERIMENT FUNCTION
# -------------------------------------------------------------------

def run_experiment(model_name: str, crop: str, run_name: str):
    """
    Trains a single-crop model using the strict temporal split.
    """
    if model_name not in MODELS_TO_RUN:
        raise ValueError(f"Model '{model_name}' not recognized.")
        
    print("="*60)
    print(f"🚀 STARTING EXPERIMENT | Model: {model_name} | Crop: {crop.upper()} | Run: {run_name}")
    print("="*60)

    # --- 1. Load Data for SINGLE Crop ---
    df_y_all, dfs_x_all = load_and_combine_data([crop], ALL_COUNTRIES)
    df_y_all = df_y_all.reset_index() 
    
    # --- 2. Create Temporal Split (Train/Val/Test) ---
    print("\n[Split] Creating strict temporal split...")
    
    years = sorted(df_y_all[KEY_YEAR].unique())
    if len(years) < 6:
        raise ValueError(f"Not enough years ({len(years)}) to create a 3-year test / 3-year val split.")

    # Last 3 years for Test
    test_years = years[-3:]
    # 3 years before that for Validation
    val_years = years[-6:-3]
    # Everything else for Train
    train_years = years[:-6]

    print(f" Train Years: {train_years}")
    print(f" Val Years:   {val_years}")
    print(f" Test Years:  {test_years}")

    df_y_train = df_y_all[df_y_all[KEY_YEAR].isin(train_years)].set_index([KEY_LOC, KEY_YEAR])
    df_y_val = df_y_all[df_y_all[KEY_YEAR].isin(val_years)].set_index([KEY_LOC, KEY_YEAR])
    df_y_test = df_y_all[df_y_all[KEY_YEAR].isin(test_years)].set_index([KEY_LOC, KEY_YEAR])
    
    print(f"[Split] Total samples: {len(df_y_all)}")
    print(f"[Split] Train samples: {len(df_y_train)}")
    print(f"[Split] Val samples:   {len(df_y_val)}")
    print(f"[Split] Test samples:  {len(df_y_test)}")
    
    # --- 3. Prepare Data & Create Datasets ---
    print("\n[Data] Preparing and normalizing data...")
    ts_scaler = StandardScaler()
    static_scaler = StandardScaler()
    
    X_ts_train, X_static_train, y_train, crops_train, regions_train, _ = prepare_data(
        df_y_train, dfs_x_all, ADM_TO_ID_MAPPING, 
        ts_scaler, static_scaler, MAX_SEQ_LEN, fit_scalers=True
    )
    train_dataset = SeqYieldDataset(X_ts_train, X_static_train, y_train, crops_train, regions_train)
    
    X_ts_val, X_static_val, y_val, crops_val, regions_val, _ = prepare_data(
        df_y_val, dfs_x_all, ADM_TO_ID_MAPPING, 
        ts_scaler, static_scaler, MAX_SEQ_LEN, fit_scalers=False
    )
    val_dataset = SeqYieldDataset(X_ts_val, X_static_val, y_val, crops_val, regions_val)
    
    X_ts_test, X_static_test, y_test, crops_test, regions_test, years_test = prepare_data(
        df_y_test, dfs_x_all, ADM_TO_ID_MAPPING, 
        ts_scaler, static_scaler, MAX_SEQ_LEN, fit_scalers=False
    )
    
    test_adm_ids = df_y_test.index.get_level_values(KEY_LOC)
    df_test_eval = pd.DataFrame({
        'adm_id': test_adm_ids,
        'year': years_test,
        'crop': crops_test,
        'region_id': regions_test,
        'y_true': y_test
    })
    df_test_eval['country'] = df_test_eval['adm_id'].str[:2].str.upper()

    test_dataset = SeqYieldDataset(X_ts_test, X_static_test, y_test, crops_test, regions_test, years_test)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # --- 4. Initialize Model ---
    model_init_kwargs = {
        "n_ts_inputs": N_TIME_SERIES_INPUTS,
        "n_static_inputs": N_STATIC_INPUTS,
        "num_regions": NUM_REGIONS,
        "d_model": D_MODEL_EMB,
        "max_seq_len": MAX_SEQ_LEN,
        "run_name": run_name,
        "save_top_k": SAVE_TOP_K_CHECKPOINTS
    }
    
    MODEL_CONSTRUCTORS = {
        "CNN": CNNModel,
        "BiLSTM": BiLSTMModel,
        "SerialCNNBiLSTM": SerialCNNBiLSTM,
        "ParallelCNNBiLSTM": ParallelCNNBiLSTM,
    }

    print(f"\n[Model] Initializing {model_name}...")
    model = MODEL_CONSTRUCTORS[model_name](**model_init_kwargs)
    
    # --- 5. Train Model ---
    model.fit(
        train_loader, 
        val_loader, 
        epochs=NN_MODELS_EPOCHS, 
        device=DEVICE,
        use_wandb=True,
        early_stopping_patience=EARLY_STOPPING_PATIENCE
    )
    
    # --- 6. Evaluate Model ---
    print("\n[Evaluate] Evaluating model on test set...")
    y_pred, y_true = model.predict(test_loader, DEVICE)
    df_test_eval['y_pred'] = y_pred
    
    # --- 7. Save Detailed Results ---
    print("[Evaluate] Saving detailed evaluation CSVs...")
    
    raw_csv_path = f"{run_name}_predictions_raw.csv"
    df_test_eval.to_csv(raw_csv_path, index=False)
    
    def evaluate_group(group, group_cols):
        metrics = calculate_metrics(group['y_true'], group['y_pred'])
        if isinstance(group_cols, str):
            metrics[group_cols] = group.name
        elif len(group_cols) > 1:
            for i, col in enumerate(group_cols):
                metrics[col] = group.name[i]
        return pd.Series(metrics)

    results_per_year = df_test_eval.groupby('year').apply(
        lambda g: evaluate_group(g, 'year')
    ).reset_index(drop=True)
    results_per_year.to_csv(f"{run_name}_results_per_year.csv", index=False)

    results_overall = pd.Series(calculate_metrics(df_test_eval['y_true'], df_test_eval['y_pred']))
    results_overall['model'] = model_name
    results_overall['crop'] = crop
    results_overall['run_name'] = run_name
    results_overall_df = results_overall.to_frame().T
    results_overall_df.to_csv(f"{run_name}_results_overall.csv", index=False)
    
    print("\n[Evaluate] Overall Test Metrics:")
    print(results_overall_df.to_string(index=False))
    
    if wandb.run:
        wandb.log({"test_metrics_overall": results_overall.to_dict()})
        wandb.log({"test_results_per_year": wandb.Table(dataframe=results_per_year)})

    print("\n" + "="*60)
    print(f"✅ EXPERIMENT COMPLETE | {model_name} on {crop.upper()}")
    print("="*60 + "\n")
    
    return results_overall.to_dict()

# -------------------------------------------------------------------
# Part 8: RUN ALL EXPERIMENTS AND COLLECT RESULTS
# -------------------------------------------------------------------

if __name__ == '__main__':
    os.makedirs("results", exist_ok=True)
    os.environ["WANDB_LOG_SYSTEM"] = "true"
    
    all_results = []

    # Loop over CROPS first, then MODELS
    for crop in CROPS_TO_RUN:
        for model_name in tqdm.tqdm(MODELS_TO_RUN, desc=f"Models for {crop}"):
            
            run_name = f"{model_name}_{crop}_US_temporal_{NN_MODELS_EPOCHS}e"
            
            try:
                wandb.init(
                    project="cybench-yield-prediction-refactor",
                    name=run_name,
                    config={
                        "model": model_name,
                        "crop": crop,
                        "epochs": NN_MODELS_EPOCHS,
                        "batch_size": BATCH_SIZE,
                        "data_split": "temporal_last_3yr_test_3yr_val", 
                        "embedding_dim": D_MODEL_EMB,
                        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
                        "save_top_k": SAVE_TOP_K_CHECKPOINTS
                    },
                    reinit=True
                )
                
                overall_metrics = run_experiment(model_name=model_name, crop=crop, run_name=f"results/{run_name}")
                all_results.append(overall_metrics)
                
            except Exception as e:
                print(f"\n" + "!"*60)
                print(f"🚨 EXPERIMENT FAILED FOR {model_name} on {crop} 🚨")
                print(f"Error: {e}")
                import traceback
                traceback.print_exc()
                print("!"*60 + "\n")
            finally:
                if wandb.run:
                    wandb.finish()

    results_df = pd.DataFrame(all_results)
    
    print("\n" + "#"*25 + " FINAL SUMMARY " + "#"*25)
    print("All experiments completed using separate models per crop and temporal splits (US only).")
    print("#"*65 + "\n")
    
    print(results_df.to_string(index=False))
    
    summary_path = f"results/temporal_CNN_summary_results_US_{NN_MODELS_EPOCHS}e.csv"
    results_df.to_csv(summary_path, index=False)
    print(f"\nSummary results saved to {summary_path}")

Using device: cpu
Running for 10 epochs with patience 5.
Models: ['CNN', 'BiLSTM', 'SerialCNNBiLSTM', 'ParallelCNNBiLSTM']
Crops: ['maize', 'wheat']


Models for maize:   0%|          | 0/4 [00:00<?, ?it/s]wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


🚀 STARTING EXPERIMENT | Model: CNN | Crop: MAIZE | Run: results/CNN_maize_US_temporal_10e

[Load] Loading data for ['maize'] in 1 countries...
  Loading maize...
[Load] Max sequence length (dekads) found: 199
[Load] Found 2221 unique regions (adm_id).

[Split] Creating strict temporal split...
 Train Years: [np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]
 Val Years:   [np.int64(2018), np.int64(2019), np.int64(2020)]
 Test Years:  [np.int64(2021), np.int64(2022), np.int64(2023)]
[Split] Total samples: 33193
[Split] Train samples: 24580
[Split] Val samples:   4243
[Split] Test samples:  4370

[Data] Preparing and normalizing data...
[Dataset] Created: X_ts=torch.Size([24580, 199, 9]), X_static=torch.Size([24580, 2]), y=torch.Size([24580]), regions=torch.Size([24580])
[Dataset] Created: X_ts=torch.S

/var/folders/5v/801lr_c574l81hvxwx6m27pr0000gn/T/ipykernel_77921/872396729.py:657: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  results_per_year = df_test_eval.groupby('year').apply(
wandb: ERROR The nbformat package was not found. It is required to save notebook history.



✅ EXPERIMENT COMPLETE | CNN on MAIZE



...

/var/folders/5v/801lr_c574l81hvxwx6m27pr0000gn/T/ipykernel_77921/872396729.py:657: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  results_per_year = df_test_eval.groupby('year').apply(
wandb: ERROR The nbformat package was not found. It is required to save notebook history.



✅ EXPERIMENT COMPLETE | ParallelCNNBiLSTM on WHEAT



epoch,▁▂▃▃▄▅▆▆▇█
train_loss,█▄▃▂▂▁▁▁▁▁
val_loss,█▆▄▃▂▁▁▂▁▁
val_rmse,█▆▄▃▂▁▂▂▁▁
epoch,10
train_loss,0.40577
val_loss,0.47352
val_rmse,0.68813


Models for wheat: 100%|██████████| 4/4 [31:24<00:00, 471.02s/it]


######################### FINAL SUMMARY #########################
All experiments completed using separate models per crop and temporal splits (US only).
#################################################################

    rmse    nrmse       r2     mape             model  crop                                        run_name
1.972330 0.195388 0.366147 0.181906               CNN maize               results/CNN_maize_US_temporal_10e
2.064293 0.204499 0.305661 0.188639            BiLSTM maize            results/BiLSTM_maize_US_temporal_10e
2.123106 0.210325 0.265533 0.193425   SerialCNNBiLSTM maize   results/SerialCNNBiLSTM_maize_US_temporal_10e
2.146716 0.212664 0.249106 0.198002 ParallelCNNBiLSTM maize results/ParallelCNNBiLSTM_maize_US_temporal_10e
1.003181 0.236921 0.565920 0.217509               CNN wheat               results/CNN_wheat_US_temporal_10e
0.978819 0.231168 0.586747 0.215035            BiLSTM wheat            results/BiLSTM_wheat_US_temporal_10e
1.016517 0.240071 0.55

In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset as TorchDataset, DataLoader
from collections import defaultdict
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.preprocessing import StandardScaler
from typing import List, Dict, Optional, Tuple
import wandb
import os
import copy
import tqdm as tqdm

# --- NEW IMPORTS ---
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
# --- END NEW IMPORTS ---


# -------------------------------------------------------------------
# Part 1: IMPORTS FROM CYBENCH LIBRARY
# -------------------------------------------------------------------
from cybench.config import (
    KEY_LOC,
    KEY_YEAR,
    STATIC_PREDICTORS,
    TIME_SERIES_PREDICTORS,
)
from cybench.datasets.configured import load_dfs_crop

# -------------------------------------------------------------------
# Part 2: NEW & UPDATED EXPERIMENT CONFIGURATION
# -------------------------------------------------------------------
CROPS_TO_RUN = ['maize', 'wheat']
CROP_TO_ID = {"maize": 0, "wheat": 1}

# Restricted to USA only
TRAIN_COUNTRIES = ['US']
TEST_COUNTRIES = ['US']
ALL_COUNTRIES = sorted(list(set(TRAIN_COUNTRIES + TEST_COUNTRIES)))

MODELS_TO_RUN = ["Ridge", "RandomForest", "SVR", "MLP"]

NN_MODELS_EPOCHS = 10 # Used for MLPRegressor's max_iter
EARLY_STOPPING_PATIENCE = 5 # Patience for early stopping (used by MLP)
SAVE_TOP_K_CHECKPOINTS = 3 # Kept for compatibility
D_MODEL_EMB = 32 # Kept for compatibility
BATCH_SIZE = 32
DEVICE = "cpu" # Sklearn runs on CPU primarily

print(f"Using device: {DEVICE}")
print(f"Running for {NN_MODELS_EPOCHS} epochs (for MLP) with patience {EARLY_STOPPING_PATIENCE}.")
print(f"Models: {MODELS_TO_RUN}")
print(f"Crops: {CROPS_TO_RUN}")

# Globals to be populated after data loading
N_TIME_SERIES_INPUTS = len(TIME_SERIES_PREDICTORS)
N_STATIC_INPUTS = len(STATIC_PREDICTORS)
MAX_SEQ_LEN = 36 
ADM_TO_ID_MAPPING = {}
NUM_REGIONS = 0

# -------------------------------------------------------------------
# Part 3: HELPER FUNCTIONS (Data Handling)
# -------------------------------------------------------------------

class SeqYieldDataset(TorchDataset):
    """Pytorch dataset holding time-series, targets, crop IDs, region IDs."""
    def __init__(self, X_ts: np.ndarray, X_static: np.ndarray, y: np.ndarray, crops: List[str], region_ids: List[int], years: Optional[List[int]] = None):
        self.X_ts = torch.tensor(X_ts, dtype=torch.float32)
        self.X_static = torch.tensor(X_static, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.crop_ids = torch.tensor([CROP_TO_ID.get(c, 0) for c in crops], dtype=torch.long)
        self.region_ids = torch.tensor(region_ids, dtype=torch.long)
        self.years = torch.tensor(years, dtype=torch.long) if years is not None else None
        
        print(f"[Dataset] Created: X_ts={self.X_ts.shape}, X_static={self.X_static.shape}, y={self.y.shape}")

    def __len__(self): return len(self.y)
    
    def __getitem__(self, idx):
        item = (self.X_ts[idx], self.X_static[idx], self.y[idx], self.crop_ids[idx], self.region_ids[idx])
        if self.years is not None:
            return item + (self.years[idx],)
        return item

def prepare_data(
    df_y_subset: pd.DataFrame, 
    dfs_x: Dict[str, pd.DataFrame], 
    adm_to_id_map: Dict[str, int],
    ts_scaler: StandardScaler,
    static_scaler: StandardScaler,
    max_seq_len: int,
    fit_scalers: bool = False
) -> Tuple[np.ndarray, np.ndarray, np.ndarray, List[str], List[int], List[int]]:
    
    X_ts_list, X_static_list, y_list = [], [], []
    crops_list, region_ids_list, years_list = [], [], []
    
    df_static = dfs_x['static'].reindex(columns=STATIC_PREDICTORS)
    if fit_scalers:
        static_scaler.fit(df_static)
    X_static_scaled = static_scaler.transform(df_static)
    df_static_scaled = pd.DataFrame(X_static_scaled, index=df_static.index, columns=df_static.columns)

    df_ts = dfs_x['time_series'].reindex(columns=TIME_SERIES_PREDICTORS)
    all_dekads = list(range(1, 37))
    full_idx = pd.MultiIndex.from_product(
        [df_ts.index.get_level_values(KEY_LOC).unique(), 
         df_ts.index.get_level_values(KEY_YEAR).unique(), 
         all_dekads],
        names=[KEY_LOC, KEY_YEAR, 'dekad']
    )
    df_ts = df_ts.reindex(full_idx)
    df_ts = df_ts.groupby(level=[KEY_LOC, KEY_YEAR]).apply(lambda g: g.interpolate(method='linear', limit_direction='both', axis=0))
    df_ts = df_ts.fillna(0) 
    
    if fit_scalers:
        sample_size = len(df_ts)
        ts_scaler.fit(df_ts.sample(sample_size, random_state=42))
        
    X_ts_scaled = ts_scaler.transform(df_ts)
    df_ts_scaled = pd.DataFrame(X_ts_scaled, index=df_ts.index, columns=df_ts.columns)
    
    for (adm_id, year), row in df_y_subset.iterrows():
        if adm_id not in df_static_scaled.index:
            continue
        if (adm_id, year) not in df_ts_scaled.index:
            continue
            
        y_val = row['yield']
        crop_str = row['crop']
        region_id = adm_to_id_map[adm_id]
        
        static_features = df_static_scaled.loc[adm_id].values
        ts_features_df = df_ts_scaled.loc[(adm_id, year)]
        
        ts_features = ts_features_df.values
        if len(ts_features) > max_seq_len:
            ts_features = ts_features[:max_seq_len] 
        elif len(ts_features) < max_seq_len:
            pad_width = ((0, max_seq_len - len(ts_features)), (0, 0))
            ts_features = np.pad(ts_features, pad_width, 'constant', constant_values=0)
            
        X_ts_list.append(ts_features)
        X_static_list.append(static_features)
        y_list.append(y_val)
        crops_list.append(crop_str)
        region_ids_list.append(region_id)
        years_list.append(year)

    return (
        np.array(X_ts_list),
        np.array(X_static_list),
        np.array(y_list),
        crops_list,
        region_ids_list,
        years_list
    )


# -------------------------------------------------------------------
# Part 4: SKLEARN MODEL IMPLEMENTATIONS
# -------------------------------------------------------------------

class SklearnWrapperBase:
    """
    Base wrapper for sklearn models to make them compatible with the
    fit/predict loop structure.
    """
    def __init__(self, model_instance, **kwargs):
        self.model = model_instance
        self.run_name = kwargs.get("run_name", "sklearn_run")
        print(f"Initialized SklearnWrapperBase with model: {self.model}")

    def _prepare_data(self, dataset: SeqYieldDataset) -> np.ndarray:
        x_ts = dataset.X_ts.numpy()
        x_static = dataset.X_static.numpy()
        n_samples = x_ts.shape[0]
        x_ts_flat = x_ts.reshape(n_samples, -1)
        x_combined = np.concatenate([x_ts_flat, x_static], axis=1)
        return x_combined

    def fit(self, train_loader: DataLoader, val_loader: DataLoader, epochs: int, device: str, use_wandb: bool = True, early_stopping_patience: int = 5):
        print(f"\n[Fit] Preparing data for sklearn model {self.run_name}...")
        train_dataset = train_loader.dataset
        X_train = self._prepare_data(train_dataset)
        y_train = train_dataset.y.numpy().squeeze() 

        if use_wandb:
            wandb.log({"train_samples": X_train.shape[0], "features": X_train.shape[1]})

        print(f"[Fit] Starting sklearn model.fit(X_train={X_train.shape}, y_train={y_train.shape})...")
        self.model.fit(X_train, y_train)
        print(f"[Fit] Sklearn model training complete for {self.run_name}.")

    def predict(self, test_loader: DataLoader, device: str) -> Tuple[np.ndarray, np.ndarray]:
        print(f"\n[Evaluate] Preparing data for sklearn model.predict()...")
        test_dataset = test_loader.dataset
        
        X_test = self._prepare_data(test_dataset)
        y_true = test_dataset.y.numpy().squeeze() 
        
        print(f"[Evaluate] Running sklearn model.predict(X_test={X_test.shape})...")
        y_pred = self.model.predict(X_test)
        
        return y_pred, y_true

class SklearnRidge(SklearnWrapperBase):
    def __init__(self, **kwargs):
        super().__init__(model_instance=Ridge(alpha=1.0), **kwargs)

class SklearnRandomForest(SklearnWrapperBase):
    def __init__(self, **kwargs):
        model = RandomForestRegressor(random_state=42, n_jobs=-1, verbose=1)
        super().__init__(model_instance=model, **kwargs)

class SklearnSVR(SklearnWrapperBase):
    def __init__(self, **kwargs):
        model = SVR(verbose=True)
        super().__init__(model_instance=model, **kwargs)

class SklearnMLP(SklearnWrapperBase):
    def __init__(self, epochs=20, **kwargs):
        model = MLPRegressor(
            max_iter=epochs,
            random_state=42,
            early_stopping=True, 
            n_iter_no_change=EARLY_STOPPING_PATIENCE, 
            verbose=True 
        )
        super().__init__(model_instance=model, **kwargs)


# -------------------------------------------------------------------
# Part 5: METRICS AND DATA LOADING
# -------------------------------------------------------------------

def calculate_metrics(y_true, y_pred, prefix=""):
    if isinstance(y_true, pd.Series): y_true = y_true.values
    if isinstance(y_pred, pd.Series): y_pred = y_pred.values

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    mean_y_true = np.mean(y_true)
    if mean_y_true == 0:
        nrmse = np.inf
    else:
        nrmse = (rmse / mean_y_true) 
        
    return {
        f"{prefix}rmse": rmse,
        f"{prefix}nrmse": nrmse,
        f"{prefix}r2": r2,
        f"{prefix}mape": mape
    }

def load_and_combine_data(crops: List[str], countries: List[str]) -> Tuple[pd.DataFrame, Dict[str, pd.DataFrame]]:
    print(f"\n[Load] Loading data for {crops} in {len(countries)} countries...")
    df_y_list = []
    raw_dfs_from_all_crops = []
    
    for crop in crops:
        print(f"  Loading {crop}...")
        try:
            df_y_c, dfs_x_c = load_dfs_crop(crop=crop, countries=countries)
            df_y_c['crop'] = crop 
            df_y_list.append(df_y_c)
            raw_dfs_from_all_crops.append(dfs_x_c)
            
        except Exception as e:
            print(f"Warning: Could not load data for crop {crop}. Error: {e}")
            
    if not df_y_list:
        raise RuntimeError("No data loaded. Exiting.")

    df_y_all = pd.concat(df_y_list).sort_index()
    if not df_y_all.index.is_unique:
        df_y_all = df_y_all.loc[~df_y_all.index.duplicated(keep='first')]
    
    combined_raw_dfs = defaultdict(list)
    for dfs_x_c in raw_dfs_from_all_crops:
        for key, df in dfs_x_c.items():
            if df is not None and not df.empty:
                combined_raw_dfs[key].append(df)
    
    final_raw_dfs = {}
    for key, df_list in combined_raw_dfs.items():
        try:
            concatenated_df = pd.concat(df_list)
            if not concatenated_df.index.is_unique:
                concatenated_df = concatenated_df.loc[~concatenated_df.index.duplicated(keep='first')]
            final_raw_dfs[key] = concatenated_df
        except Exception as e:
            print(f"  Warning: Could not concat raw_df '{key}'. Error: {e}")

    static_dfs_to_join = []
    ts_dfs_to_join = []
    
    for key, df in final_raw_dfs.items():
        if KEY_YEAR in df.index.names:
            ts_dfs_to_join.append(df)
        elif KEY_LOC in df.index.names:
            static_dfs_to_join.append(df)

    all_locs_index = df_y_all.index.get_level_values(KEY_LOC).unique()
    if not all_locs_index.any():
         all_locs_index = pd.Index([], name=KEY_LOC)

    if static_dfs_to_join:
        df_static_all = static_dfs_to_join[0].join(static_dfs_to_join[1:], how='outer')
        df_static_all = df_static_all.loc[:, ~df_static_all.columns.duplicated(keep='first')]
        cols_to_keep = [col for col in STATIC_PREDICTORS if col in df_static_all.columns]
        dfs_x_all_static = df_static_all[cols_to_keep]
        dfs_x_all_static = dfs_x_all_static.reindex(all_locs_index)
    else:
        dfs_x_all_static = pd.DataFrame(index=all_locs_index, columns=STATIC_PREDICTORS)

    dfs_x_all_ts = None
    if ts_dfs_to_join:
        dekad_level_dfs = [df for df in ts_dfs_to_join if df.index.nlevels > 2]
        year_level_dfs = [df for df in ts_dfs_to_join if df.index.nlevels == 2]

        df_ts_dekad = None
        df_ts_year = None
        
        if dekad_level_dfs:
            df_ts_dekad = dekad_level_dfs[0].join(dekad_level_dfs[1:], how='outer')
            
        if year_level_dfs:
            df_ts_year = year_level_dfs[0].join(year_level_dfs[1:], how='outer')

        if df_ts_dekad is not None and df_ts_year is not None:
            dfs_x_all_ts = df_ts_dekad.join(df_ts_year, how='outer')
        elif df_ts_dekad is not None:
            dfs_x_all_ts = df_ts_dekad
        elif df_ts_year is not None:
            dfs_x_all_ts = df_ts_year
        
        if dfs_x_all_ts is not None:
            dfs_x_all_ts = dfs_x_all_ts.loc[:, ~dfs_x_all_ts.columns.duplicated(keep='first')]
            cols_to_keep = [col for col in TIME_SERIES_PREDICTORS if col in dfs_x_all_ts.columns]
            dfs_x_all_ts = dfs_x_all_ts[cols_to_keep]

    if dfs_x_all_ts is None:
        base_idx = df_y_all.index.unique()
        if base_idx.any():
            dfs_x_all_ts = pd.DataFrame(index=base_idx, columns=TIME_SERIES_PREDICTORS)
        else:
            dfs_x_all_ts = pd.DataFrame(columns=TIME_SERIES_PREDICTORS)
            dfs_x_all_ts.index = pd.MultiIndex(levels=[[],[]], codes=[[],[]], names=[KEY_LOC, KEY_YEAR])

    dfs_x_all = {
        'static': dfs_x_all_static,
        'time_series': dfs_x_all_ts
    }
    
    global MAX_SEQ_LEN, ADM_TO_ID_MAPPING, NUM_REGIONS
    
    if not dfs_x_all['time_series'].empty and dfs_x_all['time_series'].index.nlevels > 2:
        group_levels = dfs_x_all['time_series'].index.names[:-1]
        dekads_per_sample = dfs_x_all['time_series'].groupby(level=group_levels).size()
        
        if not dekads_per_sample.empty:
            MAX_SEQ_LEN = dekads_per_sample.max()
            print(f"[Load] Max sequence length (dekads) found: {MAX_SEQ_LEN}")

    unique_adms = df_y_all.index.get_level_values(KEY_LOC).unique()
    ADM_TO_ID_MAPPING = {adm: i for i, adm in enumerate(unique_adms)}
    NUM_REGIONS = len(unique_adms)
    print(f"[Load] Found {NUM_REGIONS} unique regions (adm_id).")
    
    return df_y_all, dfs_x_all


# -------------------------------------------------------------------
# Part 6: MAIN EXPERIMENT FUNCTION
# -------------------------------------------------------------------

def run_experiment(model_name: str, crop: str, run_name: str):
    """
    Trains a single-crop model using the strict temporal split.
    """
    if model_name not in MODELS_TO_RUN:
        raise ValueError(f"Model '{model_name}' not recognized.")
        
    print("="*60)
    print(f"🚀 STARTING EXPERIMENT | Model: {model_name} | Crop: {crop.upper()} | Run: {run_name}")
    print("="*60)

    # --- 1. Load Data for SINGLE Crop ---
    df_y_all, dfs_x_all = load_and_combine_data([crop], ALL_COUNTRIES)
    df_y_all = df_y_all.reset_index() 
    
    # --- 2. Create Temporal Split (Train/Val/Test) ---
    print("\n[Split] Creating strict temporal split...")
    
    years = sorted(df_y_all[KEY_YEAR].unique())
    if len(years) < 6:
        raise ValueError(f"Not enough years ({len(years)}) to create a 3-year test / 3-year val split.")

    # Last 3 years for Test
    test_years = years[-3:]
    # 3 years before that for Validation
    val_years = years[-6:-3]
    # Everything else for Train
    train_years = years[:-6]

    print(f" Train Years: {train_years}")
    print(f" Val Years:   {val_years}")
    print(f" Test Years:  {test_years}")

    df_y_train = df_y_all[df_y_all[KEY_YEAR].isin(train_years)].set_index([KEY_LOC, KEY_YEAR])
    df_y_val = df_y_all[df_y_all[KEY_YEAR].isin(val_years)].set_index([KEY_LOC, KEY_YEAR])
    df_y_test = df_y_all[df_y_all[KEY_YEAR].isin(test_years)].set_index([KEY_LOC, KEY_YEAR])
    
    print(f"[Split] Total samples: {len(df_y_all)}")
    print(f"[Split] Train samples: {len(df_y_train)}")
    print(f"[Split] Val samples:   {len(df_y_val)}")
    print(f"[Split] Test samples:  {len(df_y_test)}")
    
    # --- 3. Prepare Data & Create Datasets ---
    print("\n[Data] Preparing and normalizing data...")
    ts_scaler = StandardScaler()
    static_scaler = StandardScaler()
    
    X_ts_train, X_static_train, y_train, crops_train, regions_train, _ = prepare_data(
        df_y_train, dfs_x_all, ADM_TO_ID_MAPPING, 
        ts_scaler, static_scaler, MAX_SEQ_LEN, fit_scalers=True
    )
    train_dataset = SeqYieldDataset(X_ts_train, X_static_train, y_train, crops_train, regions_train)
    
    X_ts_val, X_static_val, y_val, crops_val, regions_val, _ = prepare_data(
        df_y_val, dfs_x_all, ADM_TO_ID_MAPPING, 
        ts_scaler, static_scaler, MAX_SEQ_LEN, fit_scalers=False
    )
    val_dataset = SeqYieldDataset(X_ts_val, X_static_val, y_val, crops_val, regions_val)
    
    X_ts_test, X_static_test, y_test, crops_test, regions_test, years_test = prepare_data(
        df_y_test, dfs_x_all, ADM_TO_ID_MAPPING, 
        ts_scaler, static_scaler, MAX_SEQ_LEN, fit_scalers=False
    )
    
    test_adm_ids = df_y_test.index.get_level_values(KEY_LOC)
    df_test_eval = pd.DataFrame({
        'adm_id': test_adm_ids,
        'year': years_test,
        'crop': crops_test,
        'region_id': regions_test,
        'y_true': y_test
    })
    df_test_eval['country'] = df_test_eval['adm_id'].str[:2].str.upper()

    test_dataset = SeqYieldDataset(X_ts_test, X_static_test, y_test, crops_test, regions_test, years_test)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # --- 4. Initialize Model ---
    model_init_kwargs = {
        "n_ts_inputs": N_TIME_SERIES_INPUTS,
        "n_static_inputs": N_STATIC_INPUTS,
        "num_regions": NUM_REGIONS,
        "d_model": D_MODEL_EMB,
        "max_seq_len": MAX_SEQ_LEN,
        "run_name": run_name,
        "save_top_k": SAVE_TOP_K_CHECKPOINTS,
        "epochs": NN_MODELS_EPOCHS 
    }
    
    MODEL_CONSTRUCTORS = {
        "Ridge": SklearnRidge,
        "RandomForest": SklearnRandomForest,
        "SVR": SklearnSVR,
        "MLP": SklearnMLP,
    }

    print(f"\n[Model] Initializing {model_name}...")
    model = MODEL_CONSTRUCTORS[model_name](**model_init_kwargs)
    
    # --- 5. Train Model ---
    model.fit(
        train_loader, 
        val_loader, 
        epochs=NN_MODELS_EPOCHS, 
        device=DEVICE,
        use_wandb=True,
        early_stopping_patience=EARLY_STOPPING_PATIENCE
    )
    
    # --- 6. Evaluate Model ---
    print("\n[Evaluate] Evaluating model on combined test set...")
    y_pred, y_true = model.predict(test_loader, DEVICE)
    df_test_eval['y_pred'] = y_pred
    
    # --- 7. Save Detailed Results ---
    print("[Evaluate] Saving detailed evaluation CSVs...")
    
    raw_csv_path = f"{run_name}_predictions_raw.csv"
    df_test_eval.to_csv(raw_csv_path, index=False)
    
    def evaluate_group(group, group_cols):
        metrics = calculate_metrics(group['y_true'], group['y_pred'])
        if isinstance(group_cols, str):
            metrics[group_cols] = group.name
        elif len(group_cols) > 1:
            for i, col in enumerate(group_cols):
                metrics[col] = group.name[i]
        return pd.Series(metrics)

    results_per_year = df_test_eval.groupby('year').apply(
        lambda g: evaluate_group(g, 'year')
    ).reset_index(drop=True)
    results_per_year.to_csv(f"{run_name}_results_per_year.csv", index=False)

    results_overall = pd.Series(calculate_metrics(df_test_eval['y_true'], df_test_eval['y_pred']))
    results_overall['model'] = model_name
    results_overall['crop'] = crop
    results_overall['run_name'] = run_name
    results_overall_df = results_overall.to_frame().T
    results_overall_df.to_csv(f"{run_name}_results_overall.csv", index=False)
    
    print("\n[Evaluate] Overall Test Metrics:")
    print(results_overall_df.to_string(index=False))
    
    if wandb.run:
        wandb.log(results_overall.to_dict())
        wandb.log({"test_results_per_year": wandb.Table(dataframe=results_per_year)})

    print("\n" + "="*60)
    print(f"✅ EXPERIMENT COMPLETE | {model_name} on {crop.upper()}")
    print("="*60 + "\n")
    
    return results_overall.to_dict()


# -------------------------------------------------------------------
# Part 8: RUN ALL EXPERIMENTS AND COLLECT RESULTS
# -------------------------------------------------------------------
if __name__ == '__main__':
    os.makedirs("results", exist_ok=True)
    os.environ["WANDB_LOG_SYSTEM"] = "true"
    
    all_results = []

    # Loop over CROPS first, then MODELS
    for crop in CROPS_TO_RUN:
        for model_name in tqdm.tqdm(MODELS_TO_RUN, desc=f"Models for {crop}"):
            
            run_name = f"{model_name}_{crop}_US_temporal_{NN_MODELS_EPOCHS}e"
            
            try:
                wandb.init(
                    project="cybench-STAT-temporal-trial",
                    name=run_name,
                    config={
                        "model": model_name,
                        "crop": crop,
                        "epochs": NN_MODELS_EPOCHS,
                        "batch_size": BATCH_SIZE,
                        "data_split": "temporal_last_3yr_test_3yr_val",
                        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
                    },
                    reinit=True
                )
                
                overall_metrics = run_experiment(model_name=model_name, crop=crop, run_name=f"results/{run_name}")
                all_results.append(overall_metrics)
                
            except Exception as e:
                print(f"\n" + "!"*60)
                print(f"🚨 EXPERIMENT FAILED FOR {model_name} on {crop} 🚨")
                print(f"Error: {e}")
                import traceback
                traceback.print_exc()
                print("!"*60 + "\n")
            finally:
                if wandb.run:
                    wandb.finish()

    results_df = pd.DataFrame(all_results)
    
    print("\n" + "#"*25 + " FINAL SUMMARY " + "#"*25)
    print("All experiments completed using separate models per crop and temporal splits (US only).")
    print("#"*65 + "\n")
    
    print(results_df.to_string(index=False))
    
    summary_path = f"results/temporal_STAT_summary_results_US_{NN_MODELS_EPOCHS}e.csv"
    results_df.to_csv(summary_path, index=False)
    print(f"\nSummary results saved to {summary_path}")

Using device: cpu
Running for 10 epochs (for MLP) with patience 5.
Models: ['Ridge', 'RandomForest', 'SVR', 'MLP']
Crops: ['maize', 'wheat']


Models for maize:   0%|          | 0/4 [00:00<?, ?it/s]

🚀 STARTING EXPERIMENT | Model: Ridge | Crop: MAIZE | Run: results/Ridge_maize_US_temporal_10e

[Load] Loading data for ['maize'] in 1 countries...
  Loading maize...
[Load] Max sequence length (dekads) found: 199
[Load] Found 2221 unique regions (adm_id).

[Split] Creating strict temporal split...
 Train Years: [np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017)]
 Val Years:   [np.int64(2018), np.int64(2019), np.int64(2020)]
 Test Years:  [np.int64(2021), np.int64(2022), np.int64(2023)]
[Split] Total samples: 33193
[Split] Train samples: 24580
[Split] Val samples:   4243
[Split] Test samples:  4370

[Data] Preparing and normalizing data...
[Dataset] Created: X_ts=torch.Size([24580, 199, 9]), X_static=torch.Size([24580, 2]), y=torch.Size([24580])
[Dataset] Created: X_ts=torch.Size([4243, 199, 9]), X_st

/var/folders/5v/801lr_c574l81hvxwx6m27pr0000gn/T/ipykernel_77921/737622792.py:517: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  results_per_year = df_test_eval.groupby('year').apply(
wandb: ERROR The nbformat package was not found. It is required to save notebook history.



✅ EXPERIMENT COMPLETE | Ridge on MAIZE



...

/opt/miniconda3/envs/amb/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (10) reached and the optimization hasn't converged yet.
  warnings.warn(
/var/folders/5v/801lr_c574l81hvxwx6m27pr0000gn/T/ipykernel_77921/737622792.py:517: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  results_per_year = df_test_eval.groupby('year').apply(
wandb: ERROR The nbformat package was not found. It is required to save notebook history.



✅ EXPERIMENT COMPLETE | MLP on WHEAT



features,▁
mape,▁
nrmse,▁
r2,▁
rmse,▁
train_samples,▁
crop,wheat
features,1739
mape,0.37422
model,MLP
nrmse,0.38963


Models for wheat: 100%|██████████| 4/4 [12:09<00:00, 182.32s/it]


######################### FINAL SUMMARY #########################
All experiments completed using separate models per crop and temporal splits (US only).
#################################################################

    rmse    nrmse        r2     mape        model  crop                                   run_name
2.901146 0.287401 -0.371414 0.274968        Ridge maize        results/Ridge_maize_US_temporal_10e
2.206326 0.218569  0.206826 0.201434 RandomForest maize results/RandomForest_maize_US_temporal_10e
2.707606 0.268228 -0.194538 0.262157          SVR maize          results/SVR_maize_US_temporal_10e
2.864473 0.283768 -0.336961 0.272585          MLP maize          results/MLP_maize_US_temporal_10e
1.670713 0.394572 -0.203967 0.381699        Ridge wheat        results/Ridge_wheat_US_temporal_10e
1.032503 0.243846  0.540174 0.223482 RandomForest wheat results/RandomForest_wheat_US_temporal_10e
1.590437 0.375613 -0.091048 0.352533          SVR wheat          results/SVR_wheat_US